In [ ]:
import traceback

from pyspark.sql import DataFrame
from pyspark.errors import AnalysisException

import sempy.fabric as fabric

In [ ]:
def get_lakehouse_abfs_path(lakehouse: str) -> str:
    
    workspace_name = notebookutils.runtime.context.get("currentWorkspaceName")
    abfs_path = f"abfss://{workspace_name}@onelake.dfs.fabric.microsoft.com/{lakehouse}.lakehouse"

    return abfs_path

In [ ]:
def load_lakehouse_table(df: DataFrame, lakehouse: str, schema: str, table: str, write_mode: str):

    """
    Writes a DataFrame to a Delta table in a Microsoft Fabric Lakehouse.

    Parameters:
    ----------
    df : DataFrame
        The DataFrame to write.
    lakehouse : str
        Name of the Lakehouse.
    schema : str
        Schema (folder) name under 'Tables'.
    table : str
        Table name.
    write_mode : str
        Write mode: ``'overwrite'``, ``'append'``, ``'error'``, or ``'ignore'``.

    Raises:
    ------
    TypeError:
        If any input is not a string.
    """
    
    if not isinstance(df, DataFrame):
        raise TypeError(f"Expected a DataFrame, but got {type(df).__name__}")

    for var_name, var_value in {"lakehouse": lakehouse, "schema": schema, "table": table, "write_mode": write_mode}.items():
        if not isinstance(var_value, str):
            raise TypeError(f"Expected '{var_name}' to be a string, but got {type(var_value).__name__}.")

    abfs_path = get_lakehouse_abfs_path(lakehouse)

    path = f"{abfs_path}/Tables/{schema}/{table}"

    try:
        df.write.format("delta").option("mergeSchema", "true").mode(write_mode).save(path)
    except Exception as e:
        error_message = f"{type(e).__name__}: {str(e)}\n{traceback.format_exc()}"
        raise RuntimeError(f"Failed to write table to {path}: {error_message}")


In [ ]:
def read_lakehouse_table(lakehouse: str, schema: str, table: str, columns: str="*") -> DataFrame:

    """
    Reads a Delta table from a Microsoft Fabric Lakehouse into a Spark DataFrame,
    optionally selecting specific columns.

    Parameters:
    ----------
    lakehouse : str
        Name of the Lakehouse.
    schema : str
        Schema (folder) name under 'Tables'.
    table : str
        Table name.
    columns : str, optional
        Columns to select from the table. Can be a single column name, multiple column names
        separated by commas, or "*" to select all columns. Defaults to "*".

    Returns:
    -------
    DataFrame
        A Spark DataFrame containing the selected columns from the Delta table.

    Raises:
    ------
    TypeError:
        If any of `lakehouse`, `schema`, `table`, or `columns` is not a string.
    ValueError:
        If one or more requested columns do not exist in the Delta table.
    RuntimeError:
        If reading the Delta table fails for other reasons.
    """



    for var_name, var_value in {"lakehouse": lakehouse, "schema": schema, "table": table, "columns": columns}.items():
        if not isinstance(var_value, str):
            raise TypeError(f"Expected '{var_name}' to be a string, but got {type(var_value).__name__}.")
 
    abfs_path = get_lakehouse_abfs_path(lakehouse)

    path = f"{abfs_path}/Tables/{schema}/{table}"

    try:
        df = spark.read.format("delta").load(path).select([ col.strip() for col in columns.split(',')])
        return df
    except AnalysisException as ae:
        error_message = f"{type(ae).__name__}: {str(ae)}\n{traceback.format_exc()}"
        raise ValueError(f"Column selection failed. One or more columns may not exist columns select {columns}: {error_message}")
    except Exception as e:
        error_message = f"{type(e).__name__}: {str(e)}\n{traceback.format_exc()}"
        raise RuntimeError(f"Failed to read Delta table from path '{path}': {error_message}")


In [ ]:
def is_table_exist(lakehouse: str, schema: str, table: str, folder: str="Tables") -> bool:
    try:

        for var_name, var_value in {"lakehouse": lakehouse, "schema": schema, "table": table, "folder": folder}.items():
            if not isinstance(var_value, str):
                raise TypeError(f"Expected '{var_name}' to be a string, but got {type(var_value).__name__}.")
        
        abfs_path = get_lakehouse_abfs_path(lakehouse)

        path = f"{abfs_path}/{folder}/{schema}/{table}"

        return mssparkutils.fs.exists(path)
    except Exception as e:

        try:
            path_info = path
        except NameError:
            path_info = f"{lakehouse}/{folder}/{schema}/{table}"

        error_message = f"{type(e).__name__}: {str(e)}\n{traceback.format_exc()}"
        error = f"Error checking existence of {path_info}': {error_message}"
        raise RuntimeError(error)